# Data Cleaning

This notebook cleans and prepares the combined Divvy trip dataset for further analysis in SQL and Power BI.
The cleaning process includes:
- removing duplicate ride IDs
- filtering the analysis period
- converting datetime columns
- calculating ride duration
- checking invalid and extreme ride durations
- creating analytical time-based features

In [1]:
import pandas as pd
from pathlib import Path

data_folder = Path.cwd()

data_parts = []

for month in range(1, 8):
    file_name = f"{month:02d}.csv"
    file_path = data_folder / file_name

    df_month = pd.read_csv(file_path)
    data_parts.append(df_month)

df = pd.concat(data_parts, ignore_index=True)

print("Raw dataset shape:")
print(df.shape)

Raw dataset shape:
(3389831, 13)


## 1. Basic Cleaning

The first cleaning stage converts date columns to datetime format, removes duplicate ride IDs, and restricts the dataset to trips starting between January and July 2026. Trips with missing station names are retained because they can still be used for time-based and customer behavior analysis.

In [2]:
# Create a working copy
df_clean = df.copy()

raw_rows = len(df_clean)

# Convert date columns
df_clean["started_at"] = pd.to_datetime(df_clean["started_at"])
df_clean["ended_at"] = pd.to_datetime(df_clean["ended_at"])

# Remove duplicate ride IDs
df_clean = df_clean.drop_duplicates(subset="ride_id", keep="first")
after_duplicates = len(df_clean)

# Keep trips starting within Jan-Jul 2026
df_clean = df_clean[
    (df_clean["started_at"] >= "2026-01-01") &
    (df_clean["started_at"] < "2026-08-01")
].copy()

after_period_filter = len(df_clean)

print(f"Raw rows: {raw_rows:,}")
print(f"Rows after removing duplicate ride IDs: {after_duplicates:,}")
print(f"Duplicate rows removed: {raw_rows - after_duplicates:,}")

print(f"\nRows after date filtering: {after_period_filter:,}")
print(f"Rows outside analysis period removed: {after_duplicates - after_period_filter:,}")

Raw rows: 3,389,831
Rows after removing duplicate ride IDs: 3,389,796
Duplicate rows removed: 35

Rows after date filtering: 3,389,773
Rows outside analysis period removed: 23


## 2. Ride Duration Validation

Ride duration was calculated from the difference between trip end and start timestamps. Before removing any records, invalid and unusually long rides were inspected to determine appropriate cleaning rules.

In [3]:
# Calculate ride duration in minutes
df_clean["ride_duration_min"] = (
    df_clean["ended_at"] - df_clean["started_at"]
).dt.total_seconds() / 60

print("Ride duration summary:")
display(
    df_clean["ride_duration_min"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
    )
)

print("\nInvalid durations (<= 0 minutes):")
print((df_clean["ride_duration_min"] <= 0).sum())

print("\nTrips longer than 12 hours:")
print((df_clean["ride_duration_min"] > 720).sum())

print("\nTrips longer than 24 hours:")
print((df_clean["ride_duration_min"] > 1440).sum())

print("\nShortest rides:")
display(
    df_clean.nsmallest(10, "ride_duration_min")[
        ["ride_id", "started_at", "ended_at", "ride_duration_min"]
    ]
)

print("\nLongest rides:")
display(
    df_clean.nlargest(10, "ride_duration_min")[
        ["ride_id", "started_at", "ended_at", "ride_duration_min"]
    ]
)

Ride duration summary:


count    3.389773e+06
mean     1.532898e+01
std      5.129811e+01
min      1.233333e-03
1%       2.188787e-01
5%       2.036417e+00
25%      5.293150e+00
50%      9.259650e+00
75%      1.621285e+01
95%      3.812330e+01
99%      8.297580e+01
99.9%    1.112816e+03
max      1.559950e+03
Name: ride_duration_min, dtype: float64


Invalid durations (<= 0 minutes):
0

Trips longer than 12 hours:
4131

Trips longer than 24 hours:
2950

Shortest rides:


,ride_id,started_at,ended_at,ride_duration_min
833278,EB2B1F09D122345A,2026-04-06 08:18:53.978,2026-04-06 08:18:54.052,0.001233
2501923,DFC7AF8011900BD4,2026-06-29 17:16:52.930,2026-06-29 17:16:53.011,0.001350
3335363,063C3EEC781DA9FF,2026-07-20 19:04:47.939,2026-07-20 19:04:48.020,0.001350
99727,C61DB28F316634CD,2026-01-27 08:17:12.995,2026-01-27 08:17:13.084,0.001483
101221,F36D4927ED508220,2026-01-28 16:14:24.928,2026-01-28 16:14:25.017,0.001483
2405636,1BB963950984AEB9,2026-06-16 07:36:55.916,2026-06-16 07:36:56.012,0.001600
3310271,784E406984178711,2026-07-22 19:10:11.997,2026-07-22 19:10:12.097,0.001667
1348335,F479DFCD19685062,2026-05-19 16:34:12.991,2026-05-19 16:34:13.093,0.001700
2434878,B24E0776A576D4AC,2026-06-20 21:46:25.957,2026-06-20 21:46:26.060,0.001717
1269394,CCDCBAF3A7AE5B8F,2026-05-15 21:41:53.958,2026-05-15 21:41:54.063,0.001750



Longest rides:


,ride_id,started_at,ended_at,ride_duration_min
406780,C82393BDD3070D5D,2026-03-07 16:07:09.988,2026-03-08 18:07:06.999,1559.950183
483941,250556580DD308BE,2026-03-07 23:51:38.812,2026-03-09 01:51:33.033,1559.903683
480467,266FF1F3D3203623,2026-03-07 12:29:39.955,2026-03-08 14:29:33.587,1559.893867
484863,F2DADD2100A676E7,2026-03-07 09:43:11.203,2026-03-08 11:43:04.158,1559.882583
406912,EB04F9C062AAAF6A,2026-03-07 09:33:18.048,2026-03-08 11:33:09.975,1559.865450
478692,7CFD1B6B5319BB28,2026-03-07 12:20:32.025,2026-03-08 14:20:23.251,1559.853767
478695,B35EFB4539BB4B2B,2026-03-07 02:41:16.669,2026-03-08 04:41:03.312,1559.777383
522957,13769591C639EB0C,2026-03-07 12:45:52.881,2026-03-08 14:45:33.445,1559.676067
362196,A679963188E41606,2026-03-07 14:40:43.466,2026-03-08 16:40:22.698,1559.653867
483261,FBBF2C15825C4E6F,2026-03-07 12:28:27.947,2026-03-08 14:28:06.578,1559.643850


### Ride Duration Findings

Typical trips are relatively short: the median ride duration is approximately 9.3 minutes, while 95% of trips last less than 38.1 minutes. The duration distribution has a strong right tail. More than 4,000 trips exceed 12 hours and almost 3,000 exceed 24 hours, while the maximum recorded duration is approximately 26 hours. Very short trips lasting only a few seconds are also present. These extreme observations will be investigated before defining duration-based cleaning rules rather than being removed automatically.

In [4]:
# Short ride thresholds
print("Trips under 1 minute:")
print((df_clean["ride_duration_min"] < 1).sum())

print("\nTrips under 2 minutes:")
print((df_clean["ride_duration_min"] < 2).sum())

print("\nTrips under 5 minutes:")
print((df_clean["ride_duration_min"] < 5).sum())


# Long ride characteristics
long_rides = df_clean[df_clean["ride_duration_min"] > 1440].copy()

print("\nTrips over 24 hours by bike type:")
display(long_rides["rideable_type"].value_counts())

print("\nTrips over 24 hours by user type:")
display(long_rides["member_casual"].value_counts())

print("\nTrips over 24 hours by start month:")
display(
    long_rides["started_at"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print("\nTrips over 24 hours by start date — top 15:")
display(
    long_rides["started_at"]
    .dt.date
    .value_counts()
    .head(15)
)

Trips under 1 minute:
89174

Trips under 2 minutes:
165401

Trips under 5 minutes:
776116

Trips over 24 hours by bike type:


rideable_type
classic_bike    2950
Name: count, dtype: int64


Trips over 24 hours by user type:


member_casual
casual    2413
member     537
Name: count, dtype: int64


Trips over 24 hours by start month:


started_at
2026-01    193
2026-02    115
2026-03    263
2026-04    354
2026-05    583
2026-06    697
2026-07    745
Freq: M, Name: count, dtype: int64


Trips over 24 hours by start date — top 15:


started_at
2026-06-06    61
2026-06-13    61
2026-07-19    56
2026-06-07    50
2026-07-25    47
2026-05-16    40
2026-06-20    40
2026-07-26    39
2026-06-27    38
2026-06-19    35
2026-05-17    34
2026-06-30    34
2026-03-14    32
2026-07-12    32
2026-07-04    32
Name: count, dtype: int64

## 3. Final Duration Filtering

For the analytical dataset, trips shorter than 1 minute and longer than 24 hours were excluded. Trips under one minute are likely to represent aborted starts, docking events, or other records that provide limited value for usage analysis. Extremely long rides above 24 hours may reflect unusual rental or docking behavior and can heavily distort average ride-duration metrics.
The original cleaned dataframe is preserved, while the duration filter is applied only to the dataset used for further analysis.

In [5]:
# Create the analytical dataset using reasonable ride duration limits
analysis_df = df_clean[
    (df_clean["ride_duration_min"] >= 1) &
    (df_clean["ride_duration_min"] <= 1440)
].copy()

print(f"Rows before duration filtering: {len(df_clean):,}")
print(f"Rows after duration filtering: {len(analysis_df):,}")
print(f"Rows removed: {len(df_clean) - len(analysis_df):,}")
print(
    f"Share removed: "
    f"{(len(df_clean) - len(analysis_df)) / len(df_clean) * 100:.2f}%"
)

Rows before duration filtering: 3,389,773
Rows after duration filtering: 3,297,649
Rows removed: 92,124
Share removed: 2.72%


## 4. Feature Engineering

Several time-based features were created from the trip start timestamp to support further analysis in SQL and Power BI. These features enable comparisons by month, weekday, hour, and weekend status.

In [7]:
# Date
analysis_df["start_date"] = analysis_df["started_at"].dt.date

# Month number and name
analysis_df["month"] = analysis_df["started_at"].dt.month
analysis_df["month_name"] = analysis_df["started_at"].dt.month_name()

# Day of week
analysis_df["day_of_week"] = analysis_df["started_at"].dt.day_name()
analysis_df["day_of_week_num"] = analysis_df["started_at"].dt.dayofweek

# Hour of day
analysis_df["start_hour"] = analysis_df["started_at"].dt.hour

# Weekend flag
analysis_df["is_weekend"] = analysis_df["day_of_week_num"] >= 5

analysis_df[
    [
        "started_at",
        "start_date",
        "month",
        "month_name",
        "day_of_week",
        "day_of_week_num",
        "start_hour",
        "is_weekend"
    ]
].head()

analysis_df["ride_duration_min"] = analysis_df[
    "ride_duration_min"
].round(2)

## 5. Final Data Quality Check

The final analytical dataset was validated before export. Duplicate ride IDs, invalid ride durations, analysis period, categorical values, and missing values were checked to ensure the dataset is suitable for SQL analysis and Power BI reporting.

In [8]:
print("Final dataset shape:")
print(analysis_df.shape)

print("\nDuplicate ride IDs:")
print(analysis_df["ride_id"].duplicated().sum())

print("\nMissing values:")
missing = analysis_df.isna().sum()
missing_percent = (missing / len(analysis_df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

display(missing_summary)

print("\nDate range:")
print(analysis_df["started_at"].min())
print(analysis_df["started_at"].max())

print("\nRide duration range:")
print(analysis_df["ride_duration_min"].min())
print(analysis_df["ride_duration_min"].max())

print("\nUser types:")
print(analysis_df["member_casual"].value_counts())

print("\nBike types:")
print(analysis_df["rideable_type"].value_counts())

Final dataset shape:
(3297649, 21)

Duplicate ride IDs:
0

Missing values:


,missing_count,missing_percent
ride_id,0,0.00
rideable_type,0,0.00
started_at,0,0.00
ended_at,0,0.00
start_station_name,642809,19.49
start_station_id,642809,19.49
end_station_name,660898,20.04
end_station_id,660898,20.04
start_lat,0,0.00
start_lng,0,0.00



Date range:
2026-01-01 00:00:50.683000
2026-07-31 23:57:02.949000

Ride duration range:
1.0
1439.46

User types:
member_casual
member    2143281
casual    1154368
Name: count, dtype: int64

Bike types:
rideable_type
electric_bike    2299955
classic_bike      997694
Name: count, dtype: int64


## 6. Export Cleaned Dataset

The final analytical dataset is exported for the next stages of the project: SQL analysis and Power BI dashboard development.

In [9]:
from pathlib import Path

output_folder = Path("data") / "processed"
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / "divvy_trips_cleaned_2026_01_07.csv"
analysis_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")
print(f"Rows exported: {len(analysis_df):,}")

Saved to: data\processed\divvy_trips_cleaned_2026_01_07.csv
Rows exported: 3,297,649
